## Real-time ingestion of raw JSON events into the Bronze landing zone

In [0]:

conn_str = "*****"
eh_namespace = "*****"
eh_name = "*****"

kafka_options = {
    "kafka.bootstrap.servers": eh_namespace, "subscribe": eh_name,
    "kafka.security.protocol": "SASL_SSL", "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{conn_str}";',
    "startingOffsets": "earliest"
}


In [0]:
bronze_path = "abfss://bronze@mykafkastorageaccount.dfs.core.windows.net/orders_json"
checkpoint_path = "abfss://bronze@mykafkastorageaccount.dfs.core.windows.net/checkpoints/streaming_v1"



In [0]:
df = spark.readStream.format("kafka").options(**kafka_options).load()

df.selectExpr("CAST(value AS STRING)").writeStream \
    .format("json") \
    .option("checkpointLocation", checkpoint_path) \
    .start(bronze_path)
